In [1]:
import os
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class Head(nn.Module):
    def __init__(self, head_size, embed_dim, block_size):
        super().__init__()
        self.key = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, embed_dim, block_size):
        super().__init__()
        self.heads = nn.ModuleList([
            Head(head_size, embed_dim, block_size) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(num_heads * head_size, embed_dim)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size):
        super().__init__()
        head_size = embed_dim // num_heads
        self.sa = MultiHeadAttention(num_heads, head_size, embed_dim, block_size)
        self.ffwd = FeedForward(embed_dim)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, block_size, num_heads, num_layers):
        super().__init__()
        self.block_size = block_size

        self.token_embedding_table = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding_table = nn.Embedding(block_size, embed_dim)
        self.blocks = nn.Sequential(*[
            Block(embed_dim, num_heads, block_size) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

        self.token_embedding_table.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(
    logits_flat,
    targets_flat,
    ignore_index=-100
)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

data_file = "input.txt"
if not os.path.exists(data_file):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, data_file)

with open(data_file, 'r', encoding='utf-8') as f:
    text = f.read()

enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode(text, allowed_special={"<|endoftext|>"})
data = torch.tensor(tokens, dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

vocab_size = 50257
block_size = 64
embed_dim = 128
num_heads = 4
num_layers = 4
batch_size = 8
max_iters = 500
learning_rate = 3e-4

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

model = GPTLanguageModel(vocab_size, embed_dim, block_size, num_heads, num_layers).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Training start...")
for it in range(max_iters):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if it % 100 == 0:
        print(f"Step {it} | Loss: {loss.item():.4f}")

checkpoint_path = "gpt_checkpoint.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item(),
}, checkpoint_path)
print(f"Model saved to {checkpoint_path}")

print("\nTesting Reloaded Model...")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

prompt = "The king said"
start_tokens = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

output_tokens = model.generate(start_tokens, max_new_tokens=30)
generated_text = enc.decode(output_tokens[0].tolist())

print("\n--- Output ---")
print(generated_text)

Training start...
Step 0 | Loss: 11.0291
Step 100 | Loss: 6.8262
Step 200 | Loss: 6.4820
Step 300 | Loss: 6.5188
Step 400 | Loss: 6.7232
Model saved to gpt_checkpoint.pt

Testing Reloaded Model...

--- Output ---
The king saidch IV the willAS with givenBR,
 my KonNor my-,. one
back hath on,:ES night world live:,


In [ ]:
%cd /content
!git clone https://github.com/Muhammad78031/GPT-from-scratch.git

In [2]:
from http.client import responses

import torch
import torch.nn.functional  as f
import tiktoken
enc=tiktoken.get_encoding(
    "gpt2"
)
def format_sft_example(prompt_text,response_text):
    promtp_format=f"<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistance\n"
    response_format=f"{response_text}<|im_end|>"
    prompt_ids=enc.encode(promtp_format,allowed_special="all")
    respose_ids=enc.encode(response_format,allowed_special="all")
    input_ids=prompt_ids+respose_ids
    targets=[-100]*len(prompt_ids)+respose_ids
    x=torch.tensor(input_ids[:-1],dtype=torch.long)
    y=torch.tensor(targets[1:],dtype=torch.long)
    return x,y
prompt="Explain gravity is one sentence."
responses="Gravity is an invisible pulling force."
x,y=format_sft_example(prompt,responses)
print(f"Input Shape:   {x.shape}")
print(f"Target Shape:  {y.shape}")
print(f"Target Values: {y.tolist()}")


Input Shape:   torch.Size([46])
Target Shape:  torch.Size([46])
Target Values: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 38, 16995, 318, 281, 14836, 10427, 2700, 29847, 91, 320, 62, 437, 91, 29]


In [3]:
sft_example=[
    (
        "explain gravity is one sentence.",
        "gravity is an invisible pulling force."
    ),
    (
        "What is python ?",
        "pyhton is programming language."
    ),
    (
        "what is neural network",
        "A neaural network is a model that learns pattern from given data"
    ),
    (
        "what is token in GPT?",
        "A token is a peice of text represented by  numerical ID"
    ),
    (
        "What is machine learning ?",
        "Machine learning allows computers to learn patterns from data."
    ),
    (
        "what is an algoritham?",
        "An algoritham is a sequence of steps used to solve a problem. "
    ),
    (
        "What is a database?",
        "A database stores and organizes information."
    ),
    (
        "What is deep learning?",
        "Deep learning uses neural networks with multiple layers."
    ),
]

In [4]:

def get_sft_batch():
    batch_x = []
    batch_y = []

    for prompt, response in sft_example:
        x, y = format_sft_example(prompt, response)
        x = x[:block_size]
        y = y[:block_size]

        batch_x.append(x)
        batch_y.append(y)

    max_len = max(len(x) for x in batch_x)

    padded_x = []
    padded_y = []

    for x, y in zip(batch_x, batch_y):
        pad_len = max_len - len(x)

        x = F.pad(x, (0, pad_len))
        y = F.pad(y, (0, pad_len))

        padded_x.append(x)
        padded_y.append(y)

    xb = torch.stack(padded_x).to(device)
    yb = torch.stack(padded_y).to(device)

    return xb, yb


In [5]:
device = next(model.parameters()).device
print(" Training start from here")
model.train()
max_iters=500
for it in range(max_iters):
  xb , yb=get_sft_batch()
  logits,loss=model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
  if it % 20 == 0:
      print(f"SFT Step {it} | Loss: {loss.item():.4f}")

print("SFT Training complete!")


 Training start from here
SFT Step 0 | Loss: 10.0862
SFT Step 20 | Loss: 6.4557
SFT Step 40 | Loss: 4.4420
SFT Step 60 | Loss: 2.9153
SFT Step 80 | Loss: 1.9312
SFT Step 100 | Loss: 1.4708
SFT Step 120 | Loss: 0.9789
SFT Step 140 | Loss: 0.4782
SFT Step 160 | Loss: 0.1717
SFT Step 180 | Loss: 0.0799
SFT Step 200 | Loss: 0.0451
SFT Step 220 | Loss: 0.0328
SFT Step 240 | Loss: 0.0268
SFT Step 260 | Loss: 0.0230
SFT Step 280 | Loss: 0.0203
SFT Step 300 | Loss: 0.0182
SFT Step 320 | Loss: 0.0166
SFT Step 340 | Loss: 0.0152
SFT Step 360 | Loss: 0.0140
SFT Step 380 | Loss: 0.0131
SFT Step 400 | Loss: 0.0122
SFT Step 420 | Loss: 0.0115
SFT Step 440 | Loss: 0.0108
SFT Step 460 | Loss: 0.0102
SFT Step 480 | Loss: 0.0097
SFT Training complete!


In [6]:
model.eval()

prompt = "What is AI?"
x, y = format_sft_example(prompt, " ")

x = x[:block_size].to(device)
x = x.unsqueeze(0)

with torch.no_grad():
    logits, _ = model(x)

next_token_logits = logits[0, -1, :]
next_token = torch.argmax(next_token_logits).item()

print(enc.decode([next_token]))

im


In [7]:
model.eval()
prompt="<|im_start|>user\nWhat is deep learning?<|im_end|>\n<|im_start|>assistant\n"
start_tokens=torch.tensor(
    enc.encode(prompt),
    dtype=torch.long,
    device=device
).unsqueeze(0)
output_tokens=model.generate(start_tokens,max_new_tokens=20)
print(enc.decode(output_tokens[0].tolist()))

<|im_start|>user
What is deep learning?<|im_end|>
<|im_start|>assistant
Deep learning uses neural networks with multiple layers.<|im_end|>>|im_end


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
%cd /content
!git clone https://github.com/Muhammad78031/GPT-from-scratch.git

/content
Cloning into 'GPT-from-scratch'...


In [2]:
!ls /content

GPT-from-scratch  sample_data


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls /content/drive/MyDrive

'1.'$'\r\n''Definition of management'$'\r\n''Management (or mana....gsheet'
'A Deeper Dive into  Development Models.pdf'
 Classroom
'Colab Notebooks'
'Document from Sarmad'
'Document from Sarmad.gdoc'
 IMG-20240927-WA0003.jpeg
 inbound3860004278291783497.jpg
 inbound4187516713603004233.jpg
'lecture 3 protyping.gslides'
'Muhammad_Sarmad_Ahmed_CV_1 .pdf'
 Muhammad_Sarmad_Ahmed_CV.pdf
'MUHAMMAD SAUD AHMED (1).pdf'
'MUHAMMAD SAUD AHMED.pdf'
'Muhammad Saud Degree Verification Done .pdf'
'My Video.mp4'
'name = Sarmad       SEAT NUMBER  EB24210106084.zip'
'OS-Assignment 1.pdf'
'SARMAD RE.pdf'
'Technical and Business Writing Portfolio - Muhammad Sarmad Ahmed.gdoc'
'Untitled form.gform'


In [5]:
!ls "/content/drive/MyDrive/Colab Notebooks"

'03_07_2025 Data Analysis - Real Estate B.ipynb'   Untitled18.ipynb
'Copy of Untitled14.ipynb'			   Untitled19.ipynb
'Copy of Untitled17.ipynb'			   Untitled1.ipynb
'Copy of Untitled19.ipynb'			   Untitled20.ipynb
'Copy of Untitled21.ipynb'			   Untitled21.ipynb
'Copy of Untitled26.ipynb'			   Untitled22.ipynb
'Copy of Welcome To Colab'			   Untitled23.ipynb
'Copy of Welcome To Colab (1)'			   Untitled24.ipynb
'Copy of Welcome To Colab (2)'			   Untitled25.ipynb
'Copy of Welcome To Colab (3)'			   Untitled26.ipynb
 Untitled0.ipynb				   Untitled2.ipynb
 Untitled10.ipynb				   Untitled3.ipynb
 Untitled11.ipynb				   Untitled4.ipynb
 Untitled12.ipynb				   Untitled5.ipynb
 Untitled13.ipynb				   Untitled6.ipynb
 Untitled14.ipynb				   Untitled7.ipynb
 Untitled15.ipynb				   Untitled8.ipynb
 Untitled16.ipynb				   Untitled9.ipynb
 Untitled17.ipynb


In [6]:
import glob
import json
import os

folder = "/content/drive/MyDrive/Colab Notebooks"

for file in glob.glob(folder + "/**/*.ipynb", recursive=True):
    try:
        with open(file, "r", encoding="utf-8") as f:
            nb = json.load(f)

        code = "\n".join(
            cell.get("source", "") if isinstance(cell.get("source"), str)
            else "".join(cell.get("source", []))
            for cell in nb.get("cells", [])
            if cell.get("cell_type") == "code"
        )

        if any(word in code for word in [
            "GPTLanguageModel", "CausalSelfAttention",
            "token_embedding_table", "class Head"
        ]):
            print("GPT CODE MILA:", os.path.basename(file))
    except Exception:
        pass

GPT CODE MILA: Untitled25.ipynb
GPT CODE MILA: Untitled20.ipynb
GPT CODE MILA: Untitled21.ipynb
GPT CODE MILA: Copy of Untitled21.ipynb
GPT CODE MILA: Untitled26.ipynb
GPT CODE MILA: Copy of Untitled26.ipynb


In [7]:
import os
import glob
from datetime import datetime

folder = "/content/drive/MyDrive/Colab Notebooks"

names = [
    "Untitled25.ipynb",
    "Untitled20.ipynb",
    "Untitled21.ipynb",
    "Copy of Untitled21.ipynb",
    "Untitled26.ipynb",
    "Copy of Untitled26.ipynb"
]

for name in names:
    path = os.path.join(folder, name)
    if os.path.exists(path):
        print(
            name,
            "| Size:", round(os.path.getsize(path) / 1024, 1), "KB",
            "| Modified:", datetime.fromtimestamp(
                os.path.getmtime(path)
            )
        )

Untitled25.ipynb | Size: 20.4 KB | Modified: 2026-10-16 20:33:50
Untitled20.ipynb | Size: 9.6 KB | Modified: 2026-09-30 13:58:13
Untitled21.ipynb | Size: 9.4 KB | Modified: 2026-10-01 17:13:37
Copy of Untitled21.ipynb | Size: 13.2 KB | Modified: 2026-10-07 08:26:10
Untitled26.ipynb | Size: 5.8 KB | Modified: 2026-10-16 19:29:37
Copy of Untitled26.ipynb | Size: 5.9 KB | Modified: 2026-10-16 19:51:46


In [8]:
import json
import os

folder = "/content/drive/MyDrive/Colab Notebooks"

names = [
    "Untitled25.ipynb",
    "Untitled26.ipynb",
    "Copy of Untitled26.ipynb"
]

for name in names:
    path = os.path.join(folder, name)

    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    code = "\n".join(
        "".join(cell.get("source", []))
        for cell in nb["cells"]
        if cell["cell_type"] == "code"
    )

    print("\n---", name, "---")
    for keyword in [
        "class GPTLanguageModel",
        "class CausalSelfAttention",
        "class Head",
        "class Block",
        "def generate",
        "cross_entropy"
    ]:
        print(keyword, ":", keyword in code)


--- Untitled25.ipynb ---
class GPTLanguageModel : True
class CausalSelfAttention : False
class Head : True
class Block : True
def generate : True
cross_entropy : True

--- Untitled26.ipynb ---
class GPTLanguageModel : False
class CausalSelfAttention : False
class Head : False
class Block : False
def generate : True
cross_entropy : True

--- Copy of Untitled26.ipynb ---
class GPTLanguageModel : False
class CausalSelfAttention : False
class Head : False
class Block : False
def generate : True
cross_entropy : True


In [9]:
import shutil
import os

src = "/content/drive/MyDrive/Colab Notebooks/Untitled25.ipynb"
dst = "/content/GPT-from-scratch/GPT-from-scratch.ipynb"

shutil.copy2(src, dst)

print("Notebook copy ho gayi:", os.path.exists(dst))

Notebook copy ho gayi: True


In [10]:
%cd /content/GPT-from-scratch

/content/GPT-from-scratch


In [11]:
!git status

On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	GPT-from-scratch.ipynb

nothing added to commit but untracked files present (use "git add" to track)


In [12]:
!git add GPT-from-scratch.ipynb

In [13]:
!git commit -m "Add GPT from scratch notebook"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@eec448d10000.(none)')


In [5]:
!git config --global user.name "Muhammad78031"
!git config --global user.email "muhammadsarmadahmed@gmail.com"

In [6]:
!git commit -m "Add GPT from scratch notebook"

fatal: not a git repository (or any of the parent directories): .git


In [7]:
%cd /content/GPT-from-scratch

/content/GPT-from-scratch


In [8]:
!git clone https://github.com/Muhammad78031/GPT-from-scratch.git

Cloning into 'GPT-from-scratch'...


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
%cd /content/GPT-from-scratch
!git config user.name "Muhammad78031"
!git config user.email "muhammadsarmadahmed@gmail.com"
!git add GPT-from-scratch.ipynb
!git commit -m "Add GPT from scratch notebook"

/content/GPT-from-scratch
fatal: pathspec 'GPT-from-scratch.ipynb' did not match any files
On branch main

Initial commit

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	GPT-from-scratch/

nothing added to commit but untracked files present (use "git add" to track)
